<a href="https://colab.research.google.com/github/MarkosTouf/BDA---Final-Project/blob/BDA-FP-Team3/scripts/part3_spark_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BDA Project Part 3: PySpark Analytics in Google Colab

In [8]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BDA_Project_Part03_Pyspark_Analytics_in_Google_Colab")
    .master("local[*]")
    .getOrCreate()
)

print("Spark Session created for BDA Project Part 3 Spark Analytics.")

Spark Session created for BDA Project Part 3 Spark Analytics.


In [9]:
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

cleaned_market_data_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("interval", StringType(), True),
    StructField("open_time", TimestampType(), True),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("close_time", TimestampType(), True),
    StructField("quote_volume", DoubleType(), True),
    StructField("trade_count", DoubleType(), True),
    StructField("taker_buy_base_volume", DoubleType(), True),
    StructField("taker_buy_quote_volume", DoubleType(), True),
    StructField("price_range", DoubleType(), True),
    StructField("price_change", DoubleType(), True),
    StructField("percent_change", DoubleType(), True),
    StructField("candle_direction", StringType(), True),
])

In [16]:
# Team 3 Task 1

print("---- Team 3 Task 1 ----")
print()

cleaned_market_data_path = "/cleaned_market_data.csv"

cleaned_market_data_df = (
    spark.read
    .option("header", True)
    .schema(cleaned_market_data_schema)
    .csv(cleaned_market_data_path)
)

print(f"Loaded file: {cleaned_market_data_path}")

---- Team 3 Task 1 ----

Loaded file: /cleaned_market_data.csv


In [18]:

cleaned_market_data_df_row_count = cleaned_market_data_df.count()
cleaned_market_data_df_columns = cleaned_market_data_df.columns
cleaned_market_data_df_schema = cleaned_market_data_df.schema

print()
print(f"Number of rows: {cleaned_market_data_df_row_count}")
print(f"Cleaned marked data columns: {", ".join(cleaned_market_data_df_columns)}")

print()
print("Cleaned market data schema below:")
cleaned_market_data_df.printSchema()




Number of rows: 8364
Cleaned marked data columns: symbol, interval, open_time, open, high, low, close, volume, close_time, quote_volume, trade_count, taker_buy_base_volume, taker_buy_quote_volume, price_range, price_change, percent_change, candle_direction

Cleaned market data schema below:
root
 |-- symbol: string (nullable = true)
 |-- interval: string (nullable = true)
 |-- open_time: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- close_time: timestamp (nullable = true)
 |-- quote_volume: double (nullable = true)
 |-- trade_count: double (nullable = true)
 |-- taker_buy_base_volume: double (nullable = true)
 |-- taker_buy_quote_volume: double (nullable = true)
 |-- price_range: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- percent_change: double (nullable = true)
 |-- candle_direction: str

In [33]:
# Team 3 Task 2

print("---- Team 3 Task 2 ----")
print()

SQL_temp_view_name = "market_data"

cleaned_market_data_df.createOrReplaceTempView(SQL_temp_view_name)

print(f"Created a temporary view: {SQL_temp_view_name}")

# Run SQL Query on Temp View as SQL table name.

print("Running SQL Query on Temp View as SQL table name.")

market_data_rows = spark.sql("""
    SELECT count(1) as row_count
    FROM market_data
""").collect()

print(f"Test SQL query returned row count: {market_data_rows[0]["row_count"]}")

---- Team 3 Task 2 ----

Created a temporary view: market_data
Running SQL Query on Temp View as SQL table name.
Test SQL query returned row count: 8364


In [41]:
# Team 3 Task 3

print("---- Team 3 Task 3 ----")
print()

columns_to_verify = ["price_range", "price_change", "percent_change", "candle_direction"]

columns_to_verify_string = ", ".join(columns_to_verify)

print(f"Verified columns: {columns_to_verify_string}\n")

verify_calculated_row = spark.sql(f"""
    SELECT symbol, {columns_to_verify_string}
    FROM market_data
    LIMIT 5
""").collect()[0]

print("Example row for calculated columns verification:\n")
print(f"symbol={verify_calculated_row["symbol"]} price_range={verify_calculated_row["price_range"]} price_change={verify_calculated_row["price_change"]} percent_change={verify_calculated_row["percent_change"]} candle_direction={verify_calculated_row["candle_direction"]}")


---- Team 3 Task 3 ----

Verified columns: price_range, price_change, percent_change, candle_direction

Example row for calculated columns verification:

symbol=BTCUSDT price_range=190.0 price_change=-22.729999999995925 percent_change=-0.030932109802588266 candle_direction=down


In [51]:
pandas_sample_market_data_path = "/pandas_sample_results.csv"

pandas_sample_market_data_schema = cleaned_market_data_schema

pandas_sample_market_data_df = (
    spark.read
    .option("header", True)
    .schema(pandas_sample_market_data_schema)
    .csv(pandas_sample_market_data_path)
)

print(f"Below step to be used in Team 3 Task 4 comparisons to spark full market data:\n")

print(f"Loaded file for comparisons: {pandas_sample_market_data_path}")

pandas_sample_market_data_df_SQL_View_name = "pandas_sample_market_data"

pandas_sample_market_data_df.createOrReplaceTempView(pandas_sample_market_data_df_SQL_View_name)

print(f"Creating sample pandas data temporary view for comparisons to full spark market data: {pandas_sample_market_data_df_SQL_View_name}")


Below step to be used in Team 3 Task 4 comparisons to spark full market data:

Loaded file for comparisons: /pandas_sample_results.csv
Creating sample pandas data temporary view for comparisons to full spark market data: pandas_sample_market_data


In [57]:
# Team 3 Task 4

print("---- Team 3 Task 4 ----")
print()

print("Team 3 Task 4 part a is enrichment with time related calculated attributes derived from open_time:\n")

from pyspark.sql.functions import date_format, hour, to_date, col

cleaned_market_data_enriched_df = (
    cleaned_market_data_df
    .withColumn("trade_date", to_date(col("open_time")) )
    .withColumn("trade_hour", hour(col("open_time")) )
    .withColumn("day_of_week", date_format(col("open_time"),"E") )

)

time_derived_columns_added = ["trade_date", "trade_hour", "day_of_week"]

print(f"Columns that have been added to enrich dataset are: {", ".join(time_derived_columns_added)}.")
print()

cleaned_market_data_enriched_df_example_row = cleaned_market_data_enriched_df.first()


print(f"Example row after enrichment: trade_date={cleaned_market_data_enriched_df_example_row["trade_date"]}, trade_hour={cleaned_market_data_enriched_df_example_row["trade_hour"]}, day_of_week={cleaned_market_data_enriched_df_example_row["day_of_week"]}")

#

print("Team 3 Task 4 part b is comparison of full enriched market data to pandas sample data:\n")

cleaned_market_data_enriched_df_SQL_View_name = "market_data_time_enriched"

cleaned_market_data_enriched_df.createOrReplaceTempView(cleaned_market_data_enriched_df_SQL_View_name)

print(f"Created full data time enriched temporary view: {cleaned_market_data_enriched_df_SQL_View_name}")

# Run SQL Query on Time Enriched Market Data Temp View as SQL table name.

print("Query 1: Avg Close Price per Symbol")

print("Running SQL Query on Time Enriched full Market Data Temp View as SQL table name.")

print("Running SQL Query 1 - on full cleaned enriched market data - Average close price by symbol:")


# SQL Query 1:

spark.sql("""
    SELECT 'spark full data' as type, symbol, ROUND(AVG(close),2) as avg_close_price
    FROM market_data_time_enriched
    GROUP BY symbol
""").show(truncate=False)


print("Running SQL Query 1 - on pandas sample cleaned market data - Average close price by symbol:")

spark.sql("""
    SELECT 'pandas sample data' as type, symbol, ROUND(AVG(close),2) as avg_close_price
    FROM pandas_sample_market_data
    GROUP BY symbol
""").show(truncate=False)

# Sql Query 2

print("Running SQL Query 2 - on full cleaned enriched market data - Average volume by symbol:")

spark.sql("""
    SELECT 'spark full data' as type, symbol, ROUND(AVG(volume),2) as avg_volume
    FROM market_data_time_enriched
    GROUP BY symbol
""").show(truncate=False)

print("Running SQL Query 2 - on pandas sample cleaned market data - Average volume by symbol:")

spark.sql("""
    SELECT 'pandas sample data' as type, symbol, ROUND(AVG(volume),2) as avg_volume
    FROM pandas_sample_market_data
    GROUP BY symbol
""").show(truncate=False)

# Sql Query 3

print("Running SQL Query 3 - on full cleaned enriched market data - Row Counts by symbol:")

spark.sql("""
    SELECT 'spark full data' as type, symbol, count(1) as row_count
    FROM market_data_time_enriched
    GROUP BY symbol
""").show(truncate=False)

print("Running SQL Query 3 - on pandas sample cleaned market data - Row Counts by symbol:")

spark.sql("""
    SELECT 'pandas sample data' as type, symbol, count(1) as row_count
    FROM pandas_sample_market_data
    GROUP BY symbol
""").show(truncate=False)


---- Team 3 Task 4 ----

Team 3 Task 4 part a is enrichment with time related calculated attributes derived from open_time:

Columns that have been added to enrich dataset are: trade_date, trade_hour, day_of_week.

Example row after enrichment: trade_date=2026-05-29, trade_hour=23, day_of_week=Fri
Team 3 Task 4 part b is comparison of full enriched market data to pandas sample data:

Created full data time enriched temporary view: market_data_time_enriched
Query 1: Avg Close Price per Symbol
Running SQL Query on Time Enriched full Market Data Temp View as SQL table name.
Running SQL Query 1 - on full cleaned enriched market data - Average close price by symbol:
+---------------+--------+---------------+
|type           |symbol  |avg_close_price|
+---------------+--------+---------------+
|spark full data|BTCUSDT |66485.04       |
|spark full data|BNBUSDT |611.41         |
|spark full data|DOGEUSDT|0.09           |
|spark full data|DOTUSDT |1.03           |
|spark full data|LINKUSDT|8.2

Comparisons of Spark full enriched market data to Pandas Sample Market data:

As a first, the comparisons confirm that the number of records used to derive avg_price and avg_volume were different, upwards of 800 in full cleaned market data used in spark analysis and just 5 record samples in pandas sample data.

This proves that the numbers will have sample bias since the stratified sample has relevant samples for reference but does not have the completeness and scope of cleaned data that are available for analysis. Spark is using a more complete cleaned dataset to get more accurate analysis results and is using more datapoints, for more 1 hour interval data from source.

Hence the calculated average prices and volumes calculated in spark analysis with the use of a more complete dataset are more accurate and better capture the true prices and volumes captured from the market.

This highlights the advantage of spark over pandas for data analytics since its capacity of analysing larger datasets and at higher speed gives the capability for higher accuracy and analytics speed and as a result analysis of higher quality and usefulness.

In [76]:
# Team 3 Task 5

print("---- Team 3 Task 5: Volatility Ranking ----")
print()

from pyspark.sql.functions import avg, count, stddev, sum, min, max, round
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window


summary_enriched_for_volatility_ranking_df = cleaned_market_data_enriched_df.groupby("symbol").agg(
    round(avg("price_range"),3).alias("avg_price_range"),
    round(max("price_range"),3).alias("max_price_range"),
    round(min("price_range"),3).alias("min_price_range"),
    round(stddev("price_range"),3).alias("std_dev_price_range"),

)


summary_enriched_for_volatility_ranking_df.show()



---- Team 3 Task 5: Volatility Ranking ----

+--------+---------------+---------------+---------------+-------------------+
|  symbol|avg_price_range|max_price_range|min_price_range|std_dev_price_range|
+--------+---------------+---------------+---------------+-------------------+
| BTCUSDT|        443.724|        3254.17|          59.07|            315.983|
| BNBUSDT|          4.584|          40.99|           0.85|              3.521|
|DOGEUSDT|          0.001|          0.005|            0.0|              0.001|
| DOTUSDT|          0.013|          0.057|          0.003|              0.008|
|LINKUSDT|          0.084|           0.48|          0.014|              0.054|
| SOLUSDT|          0.792|           4.41|           0.19|              0.508|
| ADAUSDT|          0.002|          0.011|            0.0|              0.001|
| XRPUSDT|          0.011|          0.064|          0.003|              0.007|
|AVAXUSDT|          0.086|          0.439|          0.018|              0.056|
| ETHUS

In [75]:
# Adding the window functions for volatility ranking

avg_price_range_window = Window.orderBy(col("avg_price_range").desc())
std_dev_price_range_window = Window.orderBy(col("std_dev_price_range").desc())

ranked_volatility_df = (
  summary_enriched_for_volatility_ranking_df
  .withColumn("std_dev_price_range_rank", dense_rank().over(std_dev_price_range_window))
  .withColumn("avg_price_range_rank", dense_rank().over(avg_price_range_window))


)

# Using price standard deviation for volatility because it is part of definition of market volatility.
# As sec

print("Below Volatility Ranking: ")

ranked_volatility_df.orderBy("std_dev_price_range_rank", "avg_price_range_rank").show()

print("""
Note:
Primary Volatility Ranking measure used as proxy is std dev or price range which is part of market volatility definition.
Secondary volatility ranking measure used is avg price range.
""")


Below Volatility Ranking: 
+--------+---------------+---------------+---------------+-------------------+------------------------+--------------------+
|  symbol|avg_price_range|max_price_range|min_price_range|std_dev_price_range|std_dev_price_range_rank|avg_price_range_rank|
+--------+---------------+---------------+---------------+-------------------+------------------------+--------------------+
| BTCUSDT|        443.724|        3254.17|          59.07|            315.983|                       1|                   1|
| ETHUSDT|         15.547|         108.08|            2.4|             11.864|                       2|                   2|
| BNBUSDT|          4.584|          40.99|           0.85|              3.521|                       3|                   3|
| SOLUSDT|          0.792|           4.41|           0.19|              0.508|                       4|                   4|
|AVAXUSDT|          0.086|          0.439|          0.018|              0.056|                    

In [79]:
# Team 3 Task 6

print("---- Team 3 Task 6: Activity Ranking ----")
print()

print("Task 6 part a: Activity Ranking enrichment with measures:")
print()

summary_enriched_for_activity_ranking_df = cleaned_market_data_enriched_df.groupby("symbol").agg(
    round(sum("trade_count"),2).cast("long").alias("total_trades"),
    round(sum("quote_volume"),2).alias("total_quote_volume"),
    round(avg("volume"),2).alias("average_volume"),

)

summary_enriched_for_activity_ranking_df.show()



---- Team 3 Task 5: Activity Ranking ----

+--------+------------+------------------+--------------+
|  symbol|total_trades|total_quote_volume|average_volume|
+--------+------------+------------------+--------------+
| BTCUSDT|   131613761|     4.28759972E10|        790.22|
| BNBUSDT|    31715734|   3.89696857743E9|       7354.48|
|DOGEUSDT|    21277388|   1.89380720008E9| 2.606019728E7|
| DOTUSDT|     1387241|    2.5093034275E8|     289788.69|
|LINKUSDT|     5451600|    7.0072332144E8|     101243.48|
| SOLUSDT|    32688131|   6.92564896647E9|     113798.94|
| ADAUSDT|     4401497|   1.07006345362E9|    7016408.98|
| XRPUSDT|    24690984|   4.05229693719E9|    4138282.36|
|AVAXUSDT|     8625339|    6.8530839786E8|     115088.05|
| ETHUSDT|   124120341|  2.00681938049E10|      13601.25|
+--------+------------+------------------+--------------+



In [84]:
print("Task 6 part b: Activity Ranking application:")
print()

total_trades_window = Window.orderBy(col("total_trades").desc())
total_quote_volume_window = Window.orderBy(col("total_quote_volume").desc())
average_volume_window = Window.orderBy(col("average_volume").desc())

ranked_activity_df = (
    summary_enriched_for_activity_ranking_df
    .withColumn("total_trades_rank", dense_rank().over(total_trades_window))
    .withColumn("total_quote_volume_rank", dense_rank().over(total_quote_volume_window))
    .withColumn("average_volume_rank", dense_rank().over(average_volume_window))
)

ranked_activity_df.orderBy("total_trades_rank", "total_quote_volume_rank").show()

print("""
Note:
Primary Activity Ranking measure used as proxy is total_trades attribute.
Secondary volatility ranking measure used is total_quote_volume attribute.
Average Volume rank is used as an additional info measure.
Volume will be lower for higher price symbols but trades and quote volume in USDT show the trades activity.
""")




Task 5 part a: Activity Ranking application:

+--------+------------+------------------+--------------+-----------------+-----------------------+-------------------+
|  symbol|total_trades|total_quote_volume|average_volume|total_trades_rank|total_quote_volume_rank|average_volume_rank|
+--------+------------+------------------+--------------+-----------------+-----------------------+-------------------+
| BTCUSDT|   131613761|     4.28759972E10|        790.22|                1|                      1|                 10|
| ETHUSDT|   124120341|  2.00681938049E10|      13601.25|                2|                      2|                  8|
| SOLUSDT|    32688131|   6.92564896647E9|     113798.94|                3|                      3|                  6|
| BNBUSDT|    31715734|   3.89696857743E9|       7354.48|                4|                      5|                  9|
| XRPUSDT|    24690984|   4.05229693719E9|    4138282.36|                5|                      4|               